In [27]:
import selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import undetected_chromedriver as uc

import html
import time
from bs4 import BeautifulSoup
import re
import pandas as pd
import os
from dotenv import load_dotenv
import time
from IPython.display import clear_output

import os
import json
import requests
from urllib.parse import urljoin

BASE_URL = "https://www.lamoncloa.gob.es"
URL_INDICE = "https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23F.aspx"

HTML_DIR = "./html"
PDF_BASE_DIR = "./pdf"
JSON_OUTPUT_PATH = "./json/listado_de_archivos.json"

# Carga del sitio web principal con Selenium

Descarga del driver más reciente de 'Chrome for Testing':

https://googlechromelabs.github.io/chrome-for-testing/

In [8]:
load_dotenv("../../apis/.env")
chromedriver_path = os.getenv("CHROMEDRIVER_PATH")

service = Service(executable_path=chromedriver_path)
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=service, options=options)
# driver = uc.Chrome(headless=False) #### USING UNDETECTED CHROMEDRIVER
driver.implicitly_wait(10)
driver.get(URL_INDICE)

time.sleep(0.5)
accept_cookies_button = driver.find_element(
    By.XPATH, '//*[@id="ctl00_CookieInfo_btnLink"]'
)
accept_cookies_button.click()
time.sleep(0.5)

## Descarga del contenido del sitio web principal

In [20]:
dd23f_html = driver.page_source
print(dd23f_html[:100],"\n...")
print(dd23f_html[-100:])
# today = pd.Timestamp.today().strftime('%Y_%m_%d')
with open(os.path.join(HTML_DIR, 'dd23f_html_source.html'), 'w', encoding='utf-8') as file:
    file.write(dd23f_html)

<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="es" lang="es" class="no-js"><head id="Head"><me 
...
ookies','event_label': currentDate + ' [' + window.location.pathname + ']'});</script></body></html>


In [22]:
HTML_PATH = os.path.join(HTML_DIR, 'dd23f_html_source.html')
with open(HTML_PATH, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

In [28]:
structure = {}

# Identify the main content container by locating the first <h1>
main_h1 = soup.find("h1")
if not main_h1:
    raise ValueError("No <h1> header found in HTML.")

# Traverse following siblings to process document sections
for documents_box in main_h1.find_all_next("div", class_="documents-box"):
    h2 = documents_box.find("h2")
    if not h2:
        continue

    level_1_name = h2.get_text(strip=True)
    structure[level_1_name] = {}

    for li in documents_box.select("ul.docs-list > li"):
        h3 = li.find("h3")
        if not h3:
            continue

        level_2_name = h3.get_text(strip=True)
        structure[level_1_name][level_2_name] = []

        for link in li.find_all("a", href=True):
            href = link["href"]
            if href.lower().endswith(".pdf"):
                full_url = urljoin(BASE_URL, href)
                file_name = link.get_text(strip=True)

                structure[level_1_name][level_2_name].append({
                    "file_name": file_name,
                    "url": full_url
                })

# Save JSON structure
os.makedirs(PDF_BASE_DIR, exist_ok=True)

with open(JSON_OUTPUT_PATH, "w", encoding="utf-8") as jf:
    json.dump(structure, jf, indent=4, ensure_ascii=False)

print("Listado de archivos guardado en: ", JSON_OUTPUT_PATH)

Listado de archivos guardado en:  ./json/listado_de_archivos.json


In [4]:
driver.close()